In [0]:
storage_account = "<YOUR_STORAGE_ACCOUNT_NAME>"
storage_key = "<YOUR_STORAGE_ACCOUNT_KEY>"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
df = spark.read.parquet(
    "abfss://bronze@stalphapipeline.dfs.core.windows.net/AAPL_20260702.parquet"
)

df.show(5)

+------------------+------------------+------------------+------------------+--------+-------------------+
|             Close|              High|               Low|              Open|  Volume|               Date|
+------------------+------------------+------------------+------------------+--------+-------------------+
| 270.5074157714844|277.32473598417806| 268.5011336129586|271.75509761912826|37838100|2026-01-02 00:00:00|
| 266.7643737792969| 271.0064921177503| 265.6464557026853| 270.1381104139889|45647200|2026-01-05 00:00:00|
| 261.8734436035156| 267.0538212812338| 261.6338984265986|  266.504853430471|52352100|2026-01-06 00:00:00|
| 259.8471984863281| 263.1909919246956|259.32817380451803|262.71190159101934|48309800|2026-01-07 00:00:00|
|258.55963134765625|258.80916773471864|255.22581364132833| 256.5433578451618|50419300|2026-01-08 00:00:00|
+------------------+------------------+------------------+------------------+--------+-------------------+
only showing top 5 rows


In [0]:
df.printSchema()
df.count()

root
 |-- Close: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Date: timestamp_ntz (nullable = true)



124

In [0]:
from pyspark.sql.functions import col

silver_df = (
    df.dropDuplicates()
      .dropna()
)

silver_df.count()
silver_df.show(5)

+------------------+------------------+------------------+------------------+--------+-------------------+
|             Close|              High|               Low|              Open|  Volume|               Date|
+------------------+------------------+------------------+------------------+--------+-------------------+
| 272.6987609863281| 275.8558253670768|270.55071567552415| 274.6969199809464|32345100|2026-02-26 00:00:00|
|  264.106689453125|266.57441720150126|262.20844437502524|263.35737977838915|34203300|2026-02-18 00:00:00|
| 255.3946990966797|255.94418062939917|  253.096813183035| 253.8461228128272|40059400|2026-04-01 00:00:00|
|294.29998779296875| 301.6400146484375|294.17999267578125| 297.5400085449219|52010900|2026-06-23 00:00:00|
| 271.8895263671875| 274.6369951599546|267.46358073991865|267.61343657617186|47014600|2026-02-24 00:00:00|
+------------------+------------------+------------------+------------------+--------+-------------------+
only showing top 5 rows


In [0]:
silver_df.write.mode("overwrite").parquet(
    f"abfss://silver@{storage_account}.dfs.core.windows.net/AAPL_clean.parquet"
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, lag, col, stddev

silver_df = spark.read.parquet(
    f"abfss://silver@{storage_account}.dfs.core.windows.net/AAPL_clean.parquet"
)
window = Window.orderBy("Date")
#feature1
gold_df = silver_df.withColumn(
    "Daily_Return",
    (col("Close") - lag("Close", 1).over(window))
    / lag("Close", 1).over(window)
)
#feature2
window5 = window.rowsBetween(-4, 0)

gold_df = gold_df.withColumn(
    "MA_5",
    avg("Close").over(window5)
)
#feature3
window20 = window.rowsBetween(-19, 0)

gold_df = gold_df.withColumn(
    "MA_20",
    avg("Close").over(window20)
)
#feature4
gold_df = gold_df.withColumn(
    "Volatility",
    stddev("Daily_Return").over(window20)
)
gold_df.show(10)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------------+------------------+------------------+------------------+--------+-------------------+--------------------+------------------+------------------+--------------------+
|             Close|              High|               Low|              Open|  Volume|               Date|        Daily_Return|              MA_5|             MA_20|          Volatility|
+------------------+------------------+------------------+------------------+--------+-------------------+--------------------+------------------+------------------+--------------------+
| 270.5074157714844|277.32473598417806| 268.5011336129586|271.75509761912826|37838100|2026-01-02 00:00:00|                NULL| 270.5074157714844| 270.5074157714844|                NULL|
| 266.7643737792969| 271.0064921177503| 265.6464557026853| 270.1381104139889|45647200|2026-01-05 00:00:00|-0.01383711415641...| 268.6358947753906| 268.6358947753906|                NULL|
| 261.8734436035156| 267.0538212812338| 261.6338984265986|  266.5

In [0]:
gold_df.write.mode("overwrite").parquet(
    f"abfss://gold@{storage_account}.dfs.core.windows.net/AAPL_features.parquet"
)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
